# The Leadership Emergence Game

**Goal**: Understand how groups coordinate on leaders without explicit designation.

**Setup**: A group of *N* agents (e.g., 3-5 people) faces a series of coordination tasks that require leadership. Each round consists of:

## 1. Bidding Phase

All agents simultaneously choose an action:
- **lead**: Assert leadership (costly if rejected or if task fails)
- **follow**: Accept someone else's leadership
- **defer**: Signal "someone else should lead"

## 2. Resolution

The outcome depends on how many agents bid to lead:
- **Exactly 1 agent bids lead**: They become leader, choose group strategy
- **0 agents bid lead**: No coordination, group gets low payoff
- **2+ agents bid lead**: Conflict, medium payoff, status loss for all bidders

## 3. Task Execution

If a leader emerged, task success depends on the leader's (hidden) competence.

## Hidden Agent Types

Each agent has unobservable characteristics:
- **Dominance** $\delta_i \in [0,1]$: Intrinsic preference to lead vs. follow
- **Competence** $c_i \in [0,1]$: Actual ability to lead successfully
- **Confidence** $\phi_i \in [0,1]$: Self-assessed competence (may be miscalibrated)

## Payoffs

Each agent's utility is:

$$U_i = \text{task success value} - \text{cost of leading} \cdot \mathbb{1}(\text{led}) - \text{conflict cost} \cdot \mathbb{1}(\text{multiple leaders}) - \text{status loss} \cdot \mathbb{1}(\text{bid lead but lost}) + \text{status gain} \cdot \mathbb{1}(\text{others deferred to me})$$

## Strategic Considerations

Agents face multiple strategic tensions:
1. **Free-riding**: Following is less costly than leading (if someone else leads)
2. **Coordination failure**: If everyone defers, the group fails
3. **Conflict avoidance**: Multiple leaders cause costly conflict
4. **Information asymmetry**: Agents don't know others' competence, dominance, or confidence
5. **Reputation building**: History of successful leadership affects future coordination

## Key Research Question

Over repeated rounds, will the group converge to consistently selecting one leader? 

**Which leader emerges** - the most competent, most dominant, or most confident?

This requires agents to:
- Infer others' hidden characteristics from observed bidding behavior
- Learn from task outcomes to update beliefs about competence
- Coordinate expectations about who "should" lead without explicit communication

# Interactive Partially Observable Markov Decision Processes (I-POMDPs)

An **Interactive POMDP (I-POMDP)** extends the POMDP framework to multi-agent settings by explicitly modeling other agents in the state space. For agent *i* interacting with agents $\mathcal{J} = \{j_1, j_2, \ldots, j_{N-1}\}$, an I-POMDP is defined as:

$$\text{I-POMDP}_i = \langle IS_i, A, T_i, \Omega_i, O_i, R_i \rangle$$

where:

## Interactive State Space

**Interactive State Space (IS_i)**: $IS_i = S \times M_{j_1} \times M_{j_2} \times \cdots \times M_{j_{N-1}}$ where:
- **S**: Physical states of the environment (e.g., task outcomes, previous leadership decisions)
- **M_j**: Models of agent *j*, which can be:
  - *Subintentional models*: Simple behavioral models (e.g., always lead, always follow, random)
  - *Intentional models* (types): $\theta_j = \langle b_j, \tilde{\theta}_j \rangle$ where:
    - $b_j$ is agent *j*'s belief over $S \times M_{-j}$ (interactive state from *j*'s perspective)
    - $\tilde{\theta}_j$ represents *j*'s frame (preferences, observation function, hidden parameters)

## Key Assumptions

1. **Model Non-Manipulability (MNM)**: Agents cannot directly change other agents' models through their actions. Other agents' beliefs can only be influenced indirectly by changing the observable environment.

2. **Model Non-Observability (MNO)**: Agents cannot directly observe other agents' beliefs, preferences, or internal models. These must be inferred from observations of the physical environment and other agents' actions.

## Belief Update in Multi-Agent Settings

An agent's belief in an I-POMDP is a probability distribution over the interactive state space $IS_i = S \times \prod_{j \in \mathcal{J}} M_j$. When updating beliefs after taking action $a_i^{t-1}$ and observing $o_i^t$, agent *i* must:

1. **Predict other agents' actions**: For each agent *j*, predict $a_j^{t-1}$ based on *j*'s model $m_j^{t-1}$
2. **Update physical state beliefs**: Based on joint actions $(a_i^{t-1}, a_{-i}^{t-1})$ and observation $o_i^t$
3. **Update model beliefs**: For each agent *j*, reason about:
   - What *j* observed: $o_j^t$
   - How *j* would update its beliefs: $b_j^{t-1} \xrightarrow{a_j^{t-1}, o_j^t} b_j^t$
   - Whether *j*'s action reveals information about *j*'s type (inverse planning)

For intentional models, the belief update involves:

$$b_i^t(is^t) = \beta \sum_{is^{t-1}} \sum_{a_{-i}^{t-1}} b_i^{t-1}(is^{t-1}) \prod_{j \in \mathcal{J}} P(a_j^{t-1} | \theta_j^{t-1}) \cdot O_i(s^t, a^{t-1}, o_i^t) \cdot T_i(s^{t-1}, a^{t-1}, s^t) \cdot \prod_{j \in \mathcal{J}} \left[ \sum_{o_j^t} O_j(s^t, a^{t-1}, o_j^t) \tau_{\theta_j^t}(b_j^{t-1}, a_j^{t-1}, o_j^t, b_j^t) \right]$$

where $\tau_{\theta_j^t}$ represents *j*'s belief update and $\beta$ is a normalizing constant.

## Value Function and Solutions

The value function for an I-POMDP is defined recursively:

$$U(\theta_i) = \max_{a_i \in A_i} \left[ \sum_{is} ER_i(is, a_i) b_i(is) + \gamma \sum_{o_i \in \Omega_i} P(o_i | a_i, b_i) U(\langle SE_{\theta_i}(b_i, a_i, o_i), \tilde{\theta}_i \rangle) \right]$$

where $ER_i(is, a_i) = \sum_{a_{-i}} R_i(is, a_i, a_{-i}) \prod_{j \in \mathcal{J}} P(a_j | m_j)$ is the expected reward given other agents' models.

## Finitely Nested I-POMDPs

Since infinitely nested beliefs ("I believe that you believe that I believe...") are not computable, we construct finitely nested I-POMDPs with strategy level *l*:
- **Level 0**: POMDPs that treat other agents as part of the environment noise (no modeling of others)
- **Level 1**: Models other agents as level-0 agents (beliefs over physical states only)
- **Level *l***: Models other agents as having models up to level *l*-1

Solving an I-POMDP with strategy level *l* requires solving $O(M^l)$ POMDPs, where *M* is the number of models considered at each level.

## Computational Challenges

I-POMDPs face several computational challenges:
1. **Curse of dimensionality**: Interactive belief space grows exponentially with number of agents
2. **Model selection**: How to discretize the space of possible models for each agent
3. **Nested reasoning**: Each level of nesting requires solving lower-level decision problems
4. **Belief update complexity**: Must reason about all agents' observations and belief updates

Despite these challenges, I-POMDPs provide a principled framework for multi-agent sequential decision-making under uncertainty.

# Modeling the Leadership Emergence Game as an I-POMDP

We model the Leadership Emergence Game from agent *i*'s perspective as a level-1 I-POMDP, where *i* models other agents as having level-0 beliefs (beliefs only over physical states and hidden types, not nested beliefs about *i*).

## Interactive State Space

For a group of *N* agents, agent *i*'s interactive state space is:

$$IS_{i,1} = S \times \prod_{j \neq i} \Theta_{j,0}$$

where:

**Physical State (S)**: 
- $H$ - History of leadership bids and task outcomes from previous rounds
- Current round number $r \in \{1, 2, \ldots, R\}$

**Agent Types ($\Theta_{j,0}$)**: Each agent *j*'s level-0 type is characterized by $\theta_{j,0} = \langle b_{j,0}, \delta_j, c_j, \phi_j \rangle$:
- $b_{j,0}$ - *j*'s beliefs about other agents' types (distribution over $\{(\delta_k, c_k, \phi_k) : k \neq j\}$)
- $\delta_j \in [0,1]$ - *j*'s dominance (preference to lead)
- $c_j \in [0,1]$ - *j*'s competence (true ability)
- $\phi_j \in [0,1]$ - *j*'s confidence (self-assessed competence)

## Actions

$$A = \{\text{lead}, \text{follow}, \text{defer}\}$$

for each agent

## Observations for Agent *i*

$$\Omega_i = \underbrace{\{\text{lead}, \text{follow}, \text{defer}\}^{N-1}}_{\text{observed bids from others}} \times \underbrace{\{\text{success}, \text{failure}, \text{no leader}, \text{conflict}\}}_{\text{task outcome}}$$

Agent *i* observes:
1. **Leadership bids**: Which agents bid to lead, follow, or defer (fully observable)
2. **Task outcome**: Whether the task succeeded or failed (noisy signal of leader's competence)

Agent *i* does NOT observe:
- Other agents' types $(\delta_j, c_j, \phi_j)$
- Other agents' beliefs about types
- Other agents' reasoning processes

## Transition Function

The physical state transitions deterministically based on joint actions:

$$T(h, (a_1, \ldots, a_N), h') = \begin{cases}
1 & \text{if } h' = \text{append}(h, (a_1, \ldots, a_N, \text{outcome})) \\
0 & \text{otherwise}
\end{cases}$$

where outcome depends on how many agents bid lead:
- 1 leader: Task succeeds with probability $c_{\text{leader}}$
- 0 leaders: Automatic failure
- 2+ leaders: Conflict (partial success with probability $\frac{1}{N}\sum_j c_j$)

Hidden types $(\delta_j, c_j, \phi_j)$ remain constant across rounds.

## Observation Function

Agent *i* observes leadership bids perfectly:
$$O_i(\text{bids} | s, a, s') = \mathbb{1}[\text{bids} = a_{-i}]$$

Task outcomes are noisy signals of competence:
- If exactly one agent *j* bid lead:
  $$O_i(\text{success} | s, a, s') = c_j$$
  $$O_i(\text{failure} | s, a, s') = 1 - c_j$$
- Otherwise: Deterministic based on coordination outcome

## Reward Function

Agent *i*'s reward is:

$$R_i(s, a_i, a_{-i}) = w_1 \cdot \mathbb{1}[\text{task success}] - w_2 \cdot \mathbb{1}[a_i = \text{lead}] - w_3 \cdot \mathbb{1}[|\{j : a_j = \text{lead}\}| > 1] - w_4 \cdot \mathbb{1}[a_i = \text{lead} \land \text{lost to other}] + w_5 \cdot \mathbb{1}[a_i = \text{lead} \land \text{sole leader}]$$

where weights $(w_1, \ldots, w_5)$ can be personalized based on agent *i*'s dominance $\delta_i$:
- Higher $\delta_i$ → larger $w_5$ (status gain from leading) and smaller $w_2$ (cost of leading)

## Belief Representation and Updates

Agent *i*'s belief $b_i \in \Delta(IS_{i,1})$ is a joint distribution over:
1. **History** $h$ (fully observed, so belief is Dirac delta at current history)
2. **Other agents' types**: $P(\delta_j, c_j, \phi_j \text{ for all } j \neq i)$

For tractability, we assume **independence** across agents' types:
$$b_i(\theta_{-i}) = \prod_{j \neq i} b_i(\delta_j, c_j, \phi_j)$$

### Belief Update Process

After observing actions $a_{-i}$ and outcome $o$ at round $r$:

1. **Forward planning update** (based on task outcome):
   - If agent *j* was sole leader and task succeeded: Increase belief in high $c_j$
   - If agent *j* was sole leader and task failed: Decrease belief in high $c_j$

2. **Inverse planning update** (based on observed actions):
   - If agent *j* bid lead: Increase belief in high $\delta_j$ and high $\phi_j$
   - If agent *j* deferred: Decrease belief in high $\delta_j$
   - Magnitude of update depends on history (repeated leading → stronger signal of dominance)

Formally, using Bayes rule:
$$b_i^{r+1}(\theta_j) \propto P(a_j | \theta_j, h^r) \cdot P(o | a_j, c_j) \cdot b_i^r(\theta_j)$$

where $P(a_j | \theta_j, h^r)$ is predicted by solving agent *j*'s level-0 decision problem.

## Key Advantages of I-POMDP for Leadership Emergence

1. **Dynamic type inference**: Agents learn about others' dominance and competence from behavior patterns, not just outcomes

2. **Strategic anticipation**: Agent *i* predicts which agents will bid to lead based on:
   - Inferred dominance (preference to lead)
   - Inferred confidence (self-assessed ability)
   - History (who has led successfully before)

3. **Coordination convergence**: Over time, agents' beliefs about types align, enabling:
   - High-competence, high-dominance agents consistently bid lead
   - Others consistently defer
   - Group avoids costly conflicts and coordination failures

4. **Reputation dynamics**: Successful leadership builds reputation (beliefs about competence), which affects future coordination

5. **Handling miscalibration**: Agents can distinguish:
   - High competence + high confidence (good leaders)
   - Low competence + high confidence (overconfident, poor leaders)
   - High competence + low confidence (underconfident, missed opportunities)

## Implementation Challenges

1. **Discretization of type space**: Dominance, competence, and confidence are continuous $[0,1]$, requiring discretization for computational tractability

2. **Joint distribution complexity**: With *N* agents and *K* discrete levels per parameter, the type space has $O(K^{3(N-1)})$ combinations

3. **History representation**: Must balance history length (for learning) with computational cost

4. **Multi-agent action prediction**: Computing $P(a_{-i} | b_i)$ requires solving *N*-1 level-0 decision problems

Despite these challenges, the I-POMDP framework provides a principled way to model the emergence of leadership through strategic reasoning and belief updating in decentralized groups.